Bearing RUL Prediction: Single LSTM Model with SHAP & Integrated Gradients
Research: Cross-Domain Bearing RUL Prediction under Shaft Misalignment

In [ ]:
%pip install shap
%pip install captum

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ["OMP_NUM_THREADS"] = "1" 
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap
from captum.attr import IntegratedGradients


In [ ]:
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. GLOBAL CONFIGURATION

In [ ]:
# 1. PERBAIKI KOLOM YANG DIBUANG
class Config:
    TRAIN_DATA_PATH = r"/home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/bearing_1/bearing_1/processed_bearing1.parquet"
    TEST_DATA_PATH  = r"/home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/bearing_1/bearing_2/processed_bearing2_synthetic_20db_085.parquet"
    OUTPUT_DIR      = r"/home/praktikan/projects/github/DwiAnggara/ProyekRisetBearing/bearing_1/results_20db_085"
        
    COLS_TO_DROP = ['segment', 'time_s', 'time_min', 'bhi', 'label_vcd', 'T_cp', 'T_f',
                    'rms_ema_x', 'rms_ema_y', 'rms_ema_z']  # tambahkan ema cols juga
    TARGET_COL   = 'bhi'   # pastikan match exact dengan nama kolom parquet

    WINDOW_SIZE     = 300
    BATCH_SIZE      = 32
    HIDDEN_SIZE     = 64
    NUM_LAYERS      = 2
    LEARNING_RATE   = 0.001
    EPOCHS          = 50
    # DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BEARING_LIFE_S  = 392275

# Create output directory if it doesn't exist
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Device: {Config.DEVICE}")

In [ ]:
"""Loads parquet, scales features, and creates sequence loaders"""
print("[INFO] Loading datasets...")
print(Config.TEST_DATA_PATH)
df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
df_test  = pd.read_parquet(Config.TEST_DATA_PATH)

In [ ]:
df_train.describe()

In [ ]:
df_test.describe()

In [ ]:
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True

# 2. DATA STRUCTURES & LOADERS

In [ ]:
class BearingDataset(Dataset):
    def __init__(self, X, y, window_size, stride=1):
        self.X = X
        self.y = y
        self.window_size = window_size
        self.stride = stride
        self.indices = list(range(0, len(X) - window_size, stride))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.window_size
        return torch.tensor(self.X[start:end], dtype=torch.float32), \
               torch.tensor([self.y[end-1]], dtype=torch.float32)

def load_and_preprocess_data():
    """Loads parquet, scales features, and creates sequence loaders"""
    print("[INFO] Loading datasets...")
    print(Config.TEST_DATA_PATH)
    df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
    df_test  = pd.read_parquet(Config.TEST_DATA_PATH)
    
    # Identify feature columns
    drop_cols = Config.COLS_TO_DROP + [Config.TARGET_COL]
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    print(f"[INFO] Detected {len(feature_cols)} input features.")
    
    # Extract Target
    y_train = df_train[Config.TARGET_COL].values
    y_test  = df_test[Config.TARGET_COL].values
    
    X_train = df_train[feature_cols].values 
    X_test  = df_test[feature_cols].values

    scaler  = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    corr_with_bhi = [np.corrcoef(X_train[:, i], y_train)[0,1] for i in range(X_train.shape[1])]
    print(f"Max |corr| with BHI: {np.max(np.abs(corr_with_bhi)):.4f}")
    print(f"Mean |corr| with BHI: {np.mean(np.abs(corr_with_bhi)):.4f}")
    
    # Create DataLoaders
    train_dataset = BearingDataset(X_train, y_train, Config.WINDOW_SIZE, stride=1)
    test_dataset  = BearingDataset(X_test, y_test, Config.WINDOW_SIZE, stride=1)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, feature_cols, train_dataset, test_dataset

# 3. MODEL ARCHITECTURE


In [ ]:
class RULLSTM(nn.Module):
    def __init__(self, input_size: int):
        super(RULLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=128,  # Increased from 64
            num_layers=2,      # Increased from 2
            batch_first=True,
            dropout=0.0        # Increased dropout
        )
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_timestep = lstm_out[:, -1, :]
        out = self.relu(self.fc1(last_timestep))
        out = self.sigmoid(self.fc2(out))
        return out


# 4. TRAINING & EVALUATION FUNCTIONS


In [ ]:
def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Computes RUL/BHI evaluation metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Relative Prediction Error (RPE) & Scoring Function
    epsilon = 1e-8
    rpe = np.mean(np.abs(y_true - y_pred) / (y_true + epsilon)) * 100
    
    # PRONOSTIA Asymmetric Scoring Function
    errors = y_pred - y_true
    score = np.sum(np.where(errors < 0, np.exp(-errors/13) - 1, np.exp(errors/10) - 1))
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'RPE_pct': rpe, 'Score': score}

def train_model(model, train_loader):
    criterion = nn.MSELoss(reduction='none')  # Changed to no reduction
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)  # Higher LR
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-5)
    
    loss_history = []
    print("\n[INFO] Starting Training...")
    
    for epoch in range(Config.EPOCHS):
        model.train()
        epoch_losses = []
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(Config.DEVICE), y_batch.to(Config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            
            # Weighted loss - give more weight to degradation phase
            weights = torch.where(
                y_batch <  0.5,
                torch.tensor(3.0).to(Config.DEVICE),
                torch.tensor(1.0).to(Config.DEVICE)
            )
            loss = (criterion(outputs, y_batch) * weights).mean()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_losses.append(loss.item())
            
        avg_loss = np.mean(epoch_losses)
        loss_history.append(avg_loss)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{Config.EPOCHS}] | Loss: {avg_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
            
    return loss_history

def evaluate_and_export(model, test_loader):
    print(Config.TEST_DATA_PATH)
    """Evaluates the model and exports predictions to Excel"""
    model.eval()
    predictions, targets = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(Config.DEVICE)
            preds = model(X_batch).cpu().numpy()
            predictions.extend(preds.flatten())
            targets.extend(y_batch.numpy().flatten())
            
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    metrics = calculate_metrics(targets, predictions)
    print("\n[INFO] Evaluation Metrics on Bearing-2 (Test Set):")
    for k, v in metrics.items():
        print(f" - {k}: {v:.4f}")
        
    # Convert BHI to RUL (Simple linear conversion for evaluation)
    true_rul = targets * Config.BEARING_LIFE_S
    pred_rul = predictions * Config.BEARING_LIFE_S
    
    # Export to Excel
    df_export = pd.DataFrame({
        'Time_Step': np.arange(len(targets)),
        'True_BHI': targets,
        'Predicted_BHI': predictions,
        'True_RUL_Seconds': true_rul,
        'Predicted_RUL_Seconds': pred_rul
    })
    
    export_path = os.path.join(Config.OUTPUT_DIR, "Predictions_Results.xlsx")
    df_export.to_excel(export_path, index=False)
    print(f"[INFO] Predictions saved successfully to: {export_path}")
    
    return targets, predictions

# 5. VISUALIZATIONS & XAI (SHAP & CAPTUM)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
import torch
from captum.attr import IntegratedGradients

# ── Shared style ─────────────────────────────────────────────────────────────
PAPER_RC = {
    'font.family'     : 'DejaVu Sans',
    'font.size'       : 11,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 10,
    'ytick.labelsize' : 10,
    'legend.fontsize' : 10,
    'figure.dpi'      : 150,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'grid.linestyle'  : '--',
    'lines.linewidth' : 1.8,
}

PALETTE = {
    'true'      : '#2C3E50',
    'pred'      : '#E74C3C',
    'ig'        : '#2471A3',
    'shap'      : '#117A65',
    'error_fill': '#E74C3C',
}

TOP_N = 20   # features shown in XAI bar charts


def plot_bhi_prediction(y_true: np.ndarray, y_pred: np.ndarray, metrics: dict):
    """Publication-quality Actual vs Predicted BHI plot with metrics annotation."""
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred)

    with plt.rc_context(PAPER_RC):
        fig, ax = plt.subplots(figsize=(12, 4.5))

        ax.plot(y_true, color=PALETTE['true'], lw=2.0,  label='True BHI',      zorder=3)
        ax.plot(y_pred, color=PALETTE['pred'], lw=1.6,  label='Predicted BHI',
                linestyle='--', alpha=0.9, zorder=2)
        ax.fill_between(np.arange(len(y_true)), y_true, y_pred,
                        alpha=0.12, color=PALETTE['error_fill'], label='Prediction Error')

        metric_txt = (f"RMSE={metrics['RMSE']:.4f}  "
                      f"MAE={metrics['MAE']:.4f}  "
                      f"R²={metrics['R2']:.4f}")
        ax.set_title(f'Bearing Health Index Prediction — Test Set\n{metric_txt}',
                     fontweight='bold', pad=10)
        ax.set_xlabel('Time Step (segment index)')
        ax.set_ylabel('Health Index (BHI)')
        ax.set_ylim(-0.05, 1.12)
        ax.legend(loc='upper right', framealpha=0.9)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))

        plt.tight_layout()
        out = os.path.join(Config.OUTPUT_DIR, 'plot_bhi_prediction.png')
        plt.savefig(out, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'[INFO] BHI plot saved: {out}')


def _get_xai_tensors(test_dataset, n_bg=200, n_test=50):
    """Build properly-windowed 3D tensors from BearingDataset for XAI."""
    # BearingDataset.__getitem__ returns (window_tensor, label_tensor)
    # We need shape (N, WINDOW_SIZE, F) — must go through __getitem__
    n_avail = len(test_dataset)
    n_bg    = min(n_bg,   n_avail // 2)
    n_test  = min(n_test, n_avail - n_bg)

    bg_list   = [test_dataset[i][0] for i in range(n_bg)]
    test_list = [test_dataset[i][0] for i in range(n_bg, n_bg + n_test)]

    bg_tensor   = torch.stack(bg_list).to(Config.DEVICE)    # (n_bg,  W, F)
    test_tensor = torch.stack(test_list).to(Config.DEVICE)  # (n_test, W, F)
    return bg_tensor, test_tensor


def _bar_chart(ax, series: pd.Series, color: str, title: str, xlabel: str):
    """Horizontal bar chart with value labels — paper style."""
    bars = ax.barh(series.index, series.values,
                   color=color, edgecolor='none', height=0.65, alpha=0.88)
    ax.set_title(title, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, series.values.max() * 1.22)
    for bar, val in zip(bars, series.values):
        ax.text(val + series.values.max() * 0.015, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8.5)
    ax.tick_params(axis='y', labelsize=9)


def run_xai(model, test_dataset, feature_names):
    """SHAP + Integrated Gradients with publication-quality plots."""
    print('\n[INFO] Initializing XAI...')
    model.eval()

    bg_tensor, test_tensor = _get_xai_tensors(test_dataset)
    print(f'[INFO] XAI tensors — bg: {tuple(bg_tensor.shape)}, test: {tuple(test_tensor.shape)}')

    # ── Integrated Gradients ──────────────────────────────────────────────────
    print('[INFO] Running Integrated Gradients...')
    ig        = IntegratedGradients(model)
    baseline  = torch.zeros_like(test_tensor)
    attrs, _  = ig.attribute(test_tensor, baselines=baseline,
                              target=None, return_convergence_delta=True)
    # attrs: (N, W, F) → mean over samples and time → (F,)
    ig_importance = pd.Series(
        attrs.abs().mean(dim=(0, 1)).cpu().detach().numpy(),
        index=feature_names
    ).sort_values(ascending=True).tail(TOP_N)

    # ── SHAP GradientExplainer ────────────────────────────────────────────────
    print('[INFO] Running SHAP GradientExplainer...')

    try:
        explainer  = shap.GradientExplainer(model, bg_tensor)
        shap_vals  = explainer.shap_values(test_tensor)
        sv = shap_vals[0] if isinstance(shap_vals, list) else shap_vals

        # sv shape: (50, 300, 54, 1) -> (50, 300, 54) jika output_dim=1
        if sv.ndim == 4:
            sv = sv.squeeze(axis=-1)
        # sekarang sv 3D, aman dihitung mean pada axis=(0,1)
        shap_importance = pd.Series(
            np.abs(sv).mean(axis=(0, 1)),   # hasil 1D (54,)
            index=feature_names
        ).sort_values(ascending=True).tail(TOP_N)
    except Exception as e:
        print(f'[WARNING] SHAP failed: {e}')

    # ── Plot 1: Side-by-side IG + SHAP importance ─────────────────────────────
    with plt.rc_context(PAPER_RC):
        ncols = 2 if shap_importance is not None else 1
        fig, axes = plt.subplots(1, ncols, figsize=(8 * ncols, 8))
        if ncols == 1:
            axes = [axes]

        _bar_chart(axes[0], ig_importance,
                   color=PALETTE['ig'],
                   title=f'Integrated Gradients\nTop {TOP_N} Features',
                   xlabel='Mean |Attribution|')

        if shap_importance is not None:
            _bar_chart(axes[1], shap_importance,
                       color=PALETTE['shap'],
                       title=f'SHAP (GradientExplainer)\nTop {TOP_N} Features',
                       xlabel='Mean |SHAP Value|')

        plt.suptitle('Feature Importance Analysis — XAI on LSTM (Bearing RUL)',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        out1 = os.path.join(Config.OUTPUT_DIR, 'xai_feature_importance.png')
        plt.savefig(out1, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'[INFO] XAI importance plot saved: {out1}')

    # ── Plot 2: IG vs SHAP comparison scatter ─────────────────────────────────
    if shap_importance is not None:
        common = list(set(ig_importance.index) & set(shap_importance.index))
        ig_v   = ig_importance[common]
        sh_v   = shap_importance[common]

        ig_norm = ig_v / (ig_v.max() + 1e-12)
        sh_norm = sh_v / (sh_v.max() + 1e-12)
        corr    = np.corrcoef(ig_norm.values, sh_norm.values)[0, 1]

        with plt.rc_context(PAPER_RC):
            fig, ax = plt.subplots(figsize=(6.5, 6.5))
            ax.scatter(ig_norm.values, sh_norm.values,
                       s=70, color='#5D6D7E', edgecolors='white', linewidths=0.6, zorder=3)
            for feat, xi, yi in zip(ig_norm.index, ig_norm.values, sh_norm.values):
                ax.annotate(feat, (xi, yi), fontsize=7, alpha=0.75,
                            xytext=(4, 2), textcoords='offset points')
            lims = [0, 1.05]
            ax.plot(lims, lims, 'k--', lw=1.0, alpha=0.4, label='y = x')
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_xlabel('Normalised IG Attribution')
            ax.set_ylabel('Normalised SHAP Value')
            ax.set_title(f'IG vs SHAP Agreement (Pearson r = {corr:.3f})',
                         fontweight='bold')
            ax.legend(fontsize=9)
            plt.tight_layout()
            out2 = os.path.join(Config.OUTPUT_DIR, 'xai_ig_shap_agreement.png')
            plt.savefig(out2, dpi=300, bbox_inches='tight')
            plt.show()
            print(f'[INFO] IG vs SHAP agreement plot saved: {out2}')

    # ── Summary ───────────────────────────────────────────────────────────────
    print('\n[INFO] Top 5 features (Integrated Gradients):')
    for f, v in ig_importance.tail(5).sort_values(ascending=False).items():
        print(f'  {f:<35s} {v:.6f}')
    if shap_importance is not None:
        print('[INFO] Top 5 features (SHAP):')
        for f, v in shap_importance.tail(5).sort_values(ascending=False).items():
            print(f'  {f:<35s} {v:.6f}')

# MAIN EXECUTION


In [ ]:
if __name__ == "__main__":
    # 1. Load Data
    train_loader, test_loader, features, _, test_ds = load_and_preprocess_data()
    
    # 2. Initialize Model
    model = RULLSTM(input_size=len(features)).to(Config.DEVICE)
    
    # 3. Train
    loss_hist = train_model(model, train_loader)
    
    # 4. Evaluate & Export
    y_true, y_pred = evaluate_and_export(model, test_loader)
    metrics_dict   = calculate_metrics(y_true, y_pred)
    plot_bhi_prediction(y_true, y_pred, metrics_dict)  

    # 5. Interpret (XAI)
    run_xai(model, test_ds, features)
    
    print("\n[SUCCESS] Pipeline Executed Successfully!")

In [ ]:
# 4. Evaluate & Export
def load_and_preprocess_data():
    """Loads parquet, scales features, and creates sequence loaders"""
    print("[INFO] Loading datasets...")
    print(Config.TEST_DATA_PATH)
    df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
    df_test  = pd.read_parquet(Config.TEST_DATA_PATH)
    
    # Identify feature columns
    drop_cols = Config.COLS_TO_DROP + [Config.TARGET_COL]
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    print(f"[INFO] Detected {len(feature_cols)} input features.")
    
    # Extract Target
    y_train = df_train[Config.TARGET_COL].values
    y_test  = df_test[Config.TARGET_COL].values
    
    X_train = df_train[feature_cols].values 
    X_test  = df_test[feature_cols].values

    corr_with_bhi = [np.corrcoef(X_train[:, i], y_train)[0,1] for i in range(X_train.shape[1])]
    print(f"Max |corr| with BHI: {np.max(np.abs(corr_with_bhi)):.4f}")
    print(f"Mean |corr| with BHI: {np.mean(np.abs(corr_with_bhi)):.4f}")
    
    # Create DataLoaders
    train_dataset = BearingDataset(X_train, y_train, Config.WINDOW_SIZE, stride=1)
    test_dataset  = BearingDataset(X_test, y_test, Config.WINDOW_SIZE, stride=1)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, feature_cols, train_dataset, test_dataset

TEST_DATA_PATH  = r"D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_1\processed_bearing1.parquet"
train_loader, test_loader, features, _, test_ds = load_and_preprocess_data()
y_true, y_pred = evaluate_and_export(model, test_loader)
plot_rul(y_true, y_pred)